# 01 · Preprocessing

Inspect → harmonize → clean → optionally reserve an unseen attack → split → fit on train → transform/write each split in batches → release that split.

One dataset per run. Initial loading, splitting, and fitting still need RAM. Saved Parquet files contain ready-to-use transformed features.

## 0. Setup and Imports

In [1]:
PLATFORM = 'kaggle'  # 'colab' or 'kaggle'
assert PLATFORM in ('colab', 'kaggle')

In [2]:
if PLATFORM == 'colab':

    from google.colab import drive
    drive.mount('/content/drive')
    INPATH = '/content/drive/MyDrive/Masters/Datasets/NF-UQ-NIDS(Splitted_Dataset)'
    OUTPATH = '/content/drive/MyDrive/Masters/IDS_repo/My repo'

else : 
    INPATH = '/kaggle/input/datasets/omarahmed001/ids-datasets'
    OUTPATH = '/kaggle/working/project'
    

### Optional GPU startup

Run in a fresh kernel before sklearn/fastai imports. Missing GPU/CuPy/cuML gives a printed CPU mode; no GPU libraries are automatically installed. Compatible cuML uses info logs for actual dispatch/fallback. Other supported preprocessing estimators may also accelerate; numerical/tie differences require validation.


In [3]:
# Run first in a fresh kernel. Compatible cuML/CUDA must already be installed.
%pip install -q "scikit-learn>=1.6" category_encoders imbalanced-learn plotly
import sys
USE_GPU_NEIGHBORS = True
GPU_ACCEL_ENABLED = False
GPU_STATUS = 'CPU (GPU acceleration not requested)'
if USE_GPU_NEIGHBORS:
    try:
        import cupy as cp
        gpu_available = cp.cuda.runtime.getDeviceCount() > 0
    except Exception:
        gpu_available = False
    if not gpu_available:
        GPU_STATUS = 'CPU (NVIDIA GPU/CuPy unavailable)'
    elif 'sklearn' in sys.modules:
        raise RuntimeError('Restart the kernel and run GPU setup before sklearn/fastai imports.')
    else:
        try:
            import cuml.accel
        except ImportError:
            GPU_STATUS = 'CPU (cuML unavailable)'
        else:
            # Installation errors propagate: do not claim CPU fallback after partial activation.
            cuml.accel.install(log_level='info')
            GPU_ACCEL_ENABLED = True
            GPU_STATUS = 'cuML enabled; dispatch logs show actual GPU/CPU execution'
print(GPU_STATUS, flush=True)


Note: you may need to restart the kernel to use updated packages.
CPU (NVIDIA GPU/CuPy unavailable)


In [4]:
from pathlib import Path
from fastai.tabular.all import df_shrink
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone
from sklearn.frozen import FrozenEstimator
from sklearn.preprocessing import MinMaxScaler, QuantileTransformer
from sklearn.decomposition import PCA

def format_duration(seconds):
    tenths = int(round(float(seconds) * 10))
    hours, remaining = divmod(tenths, 36_000)
    minutes, remaining = divmod(remaining, 600)
    return f'{hours:02d}h {minutes:02d}m {remaining / 10:04.1f}s'


In [5]:
DATA_DIR = Path(INPATH)
FILES = {
    'NF_UNSW_NB15': 'NF-UNSW-NB15-v2.parquet',
    'NF_BOT_IOT': 'NF-BoT-IoT-v2.parquet',
    'NF_CSE_CIC_IDS2018': 'NF-CSE-CIC-IDS2018-v2.parquet',
    'NF_TON_IOT': 'NF-ToN-IoT-v2.parquet',
    'CIC-BoT-IoT': 'CIC-BoT-IoT-V2.parquet',
    'CSE-CIC-IDS2018': 'CSE-CIC-IDS2018.parquet',
    'UNSW-NB15': 'UNSW-NB15.parquet'
}

In [6]:
DATASET = 'CIC-BoT-IoT'  # Change this to run one of the other datasets.
HOLD_OUT_ATTACK = False  # False: include every attack class in the 70/15/15 split.
HELD_OUT_ATTACK = 'Worms'  # Used only when HOLD_OUT_ATTACK is True.
SEED = 42
MAX_PER_CLASS = 100_000  # Cap before splitting; retain every smaller-class row.


## 1. Load and inspect

In [7]:
df = pd.read_parquet(DATA_DIR / FILES[DATASET])
print(DATASET, df.shape)
display(df.head())
display(df['Attack'].value_counts())

CIC-BoT-IoT (11503556, 79)


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Attack
0,6,64465,0,2,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,1,Reconnaissance
1,6,22646986,3,1,300.0,100.0,100.0,100.0,100.0,0.0,...,3281987.0,0.0,3281987.0,3281987.0,9682500.0,5878777.50,13839423.0,5525576.0,1,DDoS
2,6,153463,1,2,0.0,100.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,1,Reconnaissance
3,17,24915470,4,1,0.0,0.0,0.0,0.0,0.0,0.0,...,4777031.0,0.0,4777031.0,4777031.0,7784938.5,2586274.75,9613711.0,5956166.0,1,DDoS
4,17,46437692,8,1,0.0,0.0,0.0,0.0,0.0,0.0,...,9597806.0,7235516.0,14714089.0,4481524.0,9080693.0,5048133.00,14796053.0,5230723.0,1,DDoS


Attack
DDoS              4786502
DoS               4781782
Reconnaissance    1896036
Benign              38435
Theft                 801
Name: count, dtype: int64

In [8]:
df.Label.value_counts()

Label
1    11465121
0       38435
Name: count, dtype: int64

In [9]:
with pd.option_context('display.max_rows', None):
    display(pd.DataFrame({
        'Dtype': df.dtypes,
        'Unique Values': df.nunique()
    }))

,Dtype,Unique Values
Protocol,int8,3
Flow Duration,int32,6106435
Total Fwd Packets,int32,1761
Total Backward Packets,int16,694
Fwd Packets Length Total,float32,6282
Bwd Packets Length Total,float32,2585
Fwd Packet Length Max,float32,876
Fwd Packet Length Min,float32,102
Fwd Packet Length Mean,float32,6752
Fwd Packet Length Std,float32,7530


- Multiple constant columns that will be dropped later

In [10]:
df['Down/Up Ratio'].value_counts()

Down/Up Ratio
0.0     9887718
1.0     1366432
2.0      242236
3.0        5192
4.0         785
7.0         460
6.0         261
5.0         223
8.0         187
9.0          52
11.0          6
10.0          3
13.0          1
Name: count, dtype: int64

### Actual sample versus the full original dataset

Keep up to `MAX_PER_CLASS = 100_000` randomly selected rows per `Attack`; retain every row from classes smaller than the cap. The cells compare this actual sample with the full original data before encoding, cleaning, splitting, or fitted transformations. No second reference sample or 50,000-row diagnostic cap is used.

The class table measures the deliberate change in class proportions. Numerical features are compared within each Attack using KS distance: 0 means matching empirical CDFs; 0.02 means a maximum CDF gap of 2 percentage points. Nominal fields use total variation distance between category frequencies (0 means matching frequencies). Missing and infinite rates are reported separately. Both distances range from 0 to 1; no universal pass/fail cutoff is imposed. Read the complete tables in `numeric_distribution_report` and `categorical_distribution_report`.

These are descriptive comparisons. KS p-values are not reported because the sample is a subset of the original, many features are discrete, and a small p-value would not establish a practically important difference. Marginal agreement does not prove matching correlations, rare subgroups, or tails. Keeping every minority row gives matching raw distributions for that class; it does not add new independent observations.

`CHECK_SAMPLE_DISTRIBUTION = False` skips the feature comparisons while retaining the class table and releasing the original. Full-data comparisons may take time; they process one column at a time and print each class. Rerun **Load and inspect** before repeating sampling.

Sampling remains before the split in this workflow: validation/test therefore reflect the capped class mix. For evaluation at the original class proportions, split the original data first and cap training only. Preprocessing is still fitted after splitting.

In [11]:
import gc

# MAX_PER_CLASS is configured above with DATASET.
CHECK_SAMPLE_DISTRIBUTION = True

# Retain the full original data until the comparison below is finished.
# This assignment does not copy the original DataFrame.
original_df = df
original_counts = original_df['Attack'].value_counts(dropna=False)

sampled_train = pd.concat(
    [
        group.sample(n=min(len(group), MAX_PER_CLASS), random_state=SEED)
        for _, group in original_df.groupby(
            'Attack', observed=True, dropna=False
        )
    ],
    ignore_index=True,
)
df = sampled_train.sample(frac=1, random_state=SEED).reset_index(drop=True)
del sampled_train

sample_counts = df['Attack'].value_counts(dropna=False)
class_comparison = pd.DataFrame({
    'Original rows': original_counts,
    'Sample rows': sample_counts,
}).fillna(0).astype('int64').rename_axis('Attack')
class_comparison['Original (%)'] = (
    100 * class_comparison['Original rows'] / len(original_df)
)
class_comparison['Sample (%)'] = (
    100 * class_comparison['Sample rows'] / len(df)
)
class_comparison['Difference (percentage points)'] = (
    class_comparison['Sample (%)'] - class_comparison['Original (%)']
)
class_comparison['Rows retained (%)'] = (
    100 * class_comparison['Sample rows'] / class_comparison['Original rows']
)
print(f'Original: {len(original_df):,} rows | Actual sample: {len(df):,} rows')
display(class_comparison.round(4))
class_tv_distance = (
    class_comparison['Sample (%)'] - class_comparison['Original (%)']
).abs().sum() / 200
print(f'Class-proportion TV distance: {class_tv_distance:.6f}')

Original: 11,503,556 rows | Actual sample: 339,236 rows


,Original rows,Sample rows,Original (%),Sample (%),Difference (percentage points),Rows retained (%)
Attack,,,,,,
Benign,38435,38435,0.3341,11.3299,10.9958,100.0000
DDoS,4786502,100000,41.6089,29.4780,-12.1309,2.0892
DoS,4781782,100000,41.5679,29.4780,-12.0899,2.0913
Reconnaissance,1896036,100000,16.4822,29.4780,12.9958,5.2742
Theft,801,801,0.0070,0.2361,0.2292,100.0000


Class-proportion TV distance: 0.242207


In [12]:
from scipy.stats import ks_2samp

# Nominal fields are compared by category frequencies, not numeric ordering.
# Add another dataset-specific nominal column here if needed.
sampling_nominal_columns = ['Protocol']

if CHECK_SAMPLE_DISTRIBUTION:
    categorical_features = [
        c for c in original_df.columns
        if c not in ('Label', 'Attack')
        and (c in sampling_nominal_columns
             or not pd.api.types.is_numeric_dtype(original_df[c]))
    ]
    numeric_features = [
        c for c in original_df.select_dtypes(include='number').columns
        if c not in ('Label', 'Attack') and c not in categorical_features
    ]

    numeric_results = []
    categorical_results = []

    for attack in original_counts.index:
        print(f'Comparing full original vs actual sample: {attack}', flush=True)
        original_mask = (original_df['Attack'].isna() if pd.isna(attack)
                         else original_df['Attack'].eq(attack))
        sample_mask = (df['Attack'].isna() if pd.isna(attack)
                       else df['Attack'].eq(attack))

        # Use every row in both datasets; process one feature at a time.
        for col in numeric_features:
            a = original_df.loc[original_mask, col].to_numpy(
                dtype=np.float64, na_value=np.nan
            )
            b = df.loc[sample_mask, col].to_numpy(
                dtype=np.float64, na_value=np.nan
            )
            finite_a = a[np.isfinite(a)]
            finite_b = b[np.isfinite(b)]

            distance = (
                ks_2samp(finite_a, finite_b, method='asymp').statistic
                if len(finite_a) and len(finite_b) else np.nan
            )
            numeric_results.append({
                'Attack': attack,
                'feature': col,
                'Original rows': len(a),
                'Sample rows': len(b),
                'KS_D': distance,
                'Original missing (%)': 100 * np.isnan(a).mean(),
                'Sample missing (%)': 100 * np.isnan(b).mean(),
                'Original infinite (%)': 100 * np.isinf(a).mean(),
                'Sample infinite (%)': 100 * np.isinf(b).mean(),
            })
            del a, b, finite_a, finite_b

        for col in categorical_features:
            original_frequencies = original_df.loc[
                original_mask, col
            ].value_counts(normalize=True, dropna=False)
            sample_frequencies = df.loc[
                sample_mask, col
            ].value_counts(normalize=True, dropna=False)
            frequencies = pd.concat([
                original_frequencies.rename('original'),
                sample_frequencies.rename('sample'),
            ], axis=1).fillna(0)
            tv_distance = (
                frequencies['original'] - frequencies['sample']
            ).abs().sum() / 2
            categorical_results.append({
                'Attack': attack,
                'feature': col,
                'TV_distance': tv_distance,
            })
            del original_frequencies, sample_frequencies, frequencies

        del original_mask, sample_mask

    numeric_distribution_report = pd.DataFrame(numeric_results, columns=[
        'Attack', 'feature', 'Original rows', 'Sample rows', 'KS_D',
        'Original missing (%)', 'Sample missing (%)',
        'Original infinite (%)', 'Sample infinite (%)',
    ]).sort_values('KS_D', ascending=False)
    categorical_distribution_report = pd.DataFrame(
        categorical_results, columns=['Attack', 'feature', 'TV_distance']
    ).sort_values('TV_distance', ascending=False)

    print('Largest numerical distribution differences (KS distance):')
    display(numeric_distribution_report.head(30))
    if not numeric_distribution_report.empty:
        display(numeric_distribution_report.groupby('Attack', dropna=False)[
            'KS_D'
        ].agg(['median', 'max']))
    print('Categorical distribution differences (total variation distance):')
    display(categorical_distribution_report)
else:
    print('Within-class distribution comparisons disabled.')

# Release the full original before encoding, splitting, and preprocessing.
# The sampled df and the small comparison reports remain available.
del original_df
gc.collect()
print('Sampling comparison finished; original data released from RAM.')

Comparing full original vs actual sample: DDoS
Comparing full original vs actual sample: DoS
Comparing full original vs actual sample: Reconnaissance
Comparing full original vs actual sample: Benign
Comparing full original vs actual sample: Theft
Largest numerical distribution differences (KS distance):


,Attack,feature,Original rows,Sample rows,KS_D,Original missing (%),Sample missing (%),Original infinite (%),Sample infinite (%)
70,DDoS,Active Max,4786502,100000,0.003717,0.0,0.0,0.0,0.0
68,DDoS,Active Mean,4786502,100000,0.003713,0.0,0.0,0.0,0.0
22,DDoS,Fwd IAT Max,4786502,100000,0.003708,0.0,0.0,0.0,0.0
71,DDoS,Active Min,4786502,100000,0.003699,0.0,0.0,0.0,0.0
35,DDoS,Fwd Packets/s,4786502,100000,0.003125,0.0,0.0,0.0,0.0
21,DDoS,Fwd IAT Std,4786502,100000,0.003063,0.0,0.0,0.0,0.0
20,DDoS,Fwd IAT Mean,4786502,100000,0.003019,0.0,0.0,0.0,0.0
23,DDoS,Fwd IAT Min,4786502,100000,0.003011,0.0,0.0,0.0,0.0
172,Reconnaissance,Fwd IAT Mean,1896036,100000,0.002981,0.0,0.0,0.0,0.0
224,Reconnaissance,Idle Mean,1896036,100000,0.002954,0.0,0.0,0.0,0.0


,median,max
Attack,,
Benign,0.000000,0.000000
DDoS,0.001430,0.003717
DoS,0.001230,0.002946
Reconnaissance,0.001135,0.002981
Theft,0.000000,0.000000


Categorical distribution differences (total variation distance):


,Attack,feature,TV_distance
1,DoS,Protocol,0.000531
0,DDoS,Protocol,0.000398
2,Reconnaissance,Protocol,0.000086
3,Benign,Protocol,0.000000
4,Theft,Protocol,0.000000


Sampling comparison finished; original data released from RAM.


In [13]:
# 1. Summarize the feature distances separately for every class.
# Multiply by 100 to express CDF gaps in percentage points.
numeric_summary = (
    numeric_distribution_report
    .groupby('Attack', observed=True, dropna=False)['KS_D']
    .agg(['median', 'max'])
    .mul(100)
    .rename(columns={
        'median': 'Median CDF gap across features (pp)',
        'max': 'Largest CDF gap (pp)',
    })
)

# Start with ALL classes, then attach their numerical results.
sample_distribution_summary = class_comparison[
    ['Original rows', 'Sample rows']
].join(numeric_summary)

print('SUMMARY — one row per class')
print('pp = percentage points; smaller gaps mean closer distributions.')

with pd.option_context(
    'display.max_rows', None,
    'display.max_columns', None,
):
    display(sample_distribution_summary.round(4))


# 2. Show five features for EACH class.
# This limits features per class, not the number of classes displayed.
for attack in sample_distribution_summary.index:
    mask = (
        numeric_distribution_report['Attack'].isna()
        if pd.isna(attack)
        else numeric_distribution_report['Attack'].eq(attack)
    )

    details = (
        numeric_distribution_report.loc[mask, ['feature', 'KS_D']]
        .sort_values('KS_D', ascending=False)
        .head(5)
        .copy()
    )
    details['Max CDF gap (pp)'] = 100 * details['KS_D']

    print(f'\n{attack} — five largest numerical feature differences:')
    display(details.reset_index(drop=True).round(6))


# 3. Show categorical results for ALL classes.
print('\nCATEGORY FREQUENCIES — zero means matching proportions.')

with pd.option_context('display.max_rows', None):
    display(
        categorical_distribution_report
        .sort_values(['Attack', 'feature'])
        .reset_index(drop=True)
        .round(6)
    )

SUMMARY — one row per class
pp = percentage points; smaller gaps mean closer distributions.


,Original rows,Sample rows,Median CDF gap across features (pp),Largest CDF gap (pp)
Attack,,,,
Benign,38435,38435,0.0000,0.0000
DDoS,4786502,100000,0.1430,0.3717
DoS,4781782,100000,0.1230,0.2946
Reconnaissance,1896036,100000,0.1135,0.2981
Theft,801,801,0.0000,0.0000



Benign — five largest numerical feature differences:


,feature,KS_D,Max CDF gap (pp)
0,Fwd Packet Length Mean,0.0,0.0
1,Fwd Packet Length Min,0.0,0.0
2,Fwd Packet Length Max,0.0,0.0
3,Bwd Packets Length Total,0.0,0.0
4,Fwd Packets Length Total,0.0,0.0



DDoS — five largest numerical feature differences:


,feature,KS_D,Max CDF gap (pp)
0,Active Max,0.003717,0.371721
1,Active Mean,0.003713,0.371297
2,Fwd IAT Max,0.003708,0.370831
3,Active Min,0.003699,0.369863
4,Fwd Packets/s,0.003125,0.312464


... (4 outputs left out)


Theft — five largest numerical feature differences:


,feature,KS_D,Max CDF gap (pp)
0,RST Flag Count,0.0,0.0
1,PSH Flag Count,0.0,0.0
2,ACK Flag Count,0.0,0.0
3,URG Flag Count,0.0,0.0
4,SYN Flag Count,0.0,0.0



CATEGORY FREQUENCIES — zero means matching proportions.


,Attack,feature,TV_distance
0,Benign,Protocol,0.000000
1,DDoS,Protocol,0.000398
2,DoS,Protocol,0.000531
3,Reconnaissance,Protocol,0.000086
4,Theft,Protocol,0.000000


## 2. Encoding

| Feature | Unique values | Type | Handling | 
|---|---:|---|---:|
| `Protocol` | 3 | Nominal | One Hot Encoding

In [14]:
col_to_one_hot = [
    'Protocol'
]

IMPUTER_FIT_ROWS = 500_000
PCA_FIT_ROWS = IMPUTER_FIT_ROWS  # None: full train; integer: sample cap.
PCA_COMPONENTS = None  # None disables PCA; e.g. 10 reduces scaled measurements to 10 columns.
NUMERIC_SCALER = QuantileTransformer(output_distribution='normal', random_state=SEED)
PREPROCESSING_VERBOSE = True
PRINT_EVERY_BATCHES = 10  # Export reporting only.


In [15]:
col_categorical = col_to_one_hot 

## 3. Dataset Shrinking

In [16]:
import warnings
from fastai.tabular.core import df_shrink_dtypes

results = []

for col in df.select_dtypes(include="number").columns:
    # Inspect one column without copying or modifying the DataFrame.
    skip = [c for c in df.columns if c != col]

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", RuntimeWarning)

        proposed = df_shrink_dtypes(
            df,
            skip=skip,
            obj2cat=False,
            int2uint=False,
        )

    if caught:
        results.append({
            "column": col,
            "original_dtype": str(df[col].dtype),
            "proposed_dtype": str(proposed.get(col, df[col].dtype)),
            "minimum": df[col].min(),
            "maximum": df[col].max(),
            "warning": "; ".join(str(w.message) for w in caught),
        })

display(pd.DataFrame(results))

""


In [17]:
df.columns = df.columns.str.strip()
df['Attack'] = df['Attack'].astype("string[pyarrow]").str.strip()

In [18]:
df = df_shrink(df, obj2cat=False, int2uint=False)
with pd.option_context('display.max_rows', None):
    display(df.dtypes)

Protocol                               int8
Flow Duration                         int32
Total Fwd Packets                     int32
Total Backward Packets                int16
Fwd Packets Length Total            float32
Bwd Packets Length Total            float32
Fwd Packet Length Max               float32
Fwd Packet Length Min               float32
Fwd Packet Length Mean              float32
Fwd Packet Length Std               float32
Bwd Packet Length Max               float32
Bwd Packet Length Min               float32
Bwd Packet Length Mean              float32
Bwd Packet Length Std               float32
Flow Bytes/s                        float32
Flow Packets/s                      float32
Flow IAT Mean                       float32
Flow IAT Std                        float32
Flow IAT Max                        float32
Flow IAT Min                        float32
Fwd IAT Total                       float32
Fwd IAT Mean                        float32
Fwd IAT Std                     

## 4. Clean

In [19]:
# Identify constant columns
constant_cols = df.columns[df.nunique(dropna=False) == 1]

print("Constant columns:", constant_cols.tolist())
print("Number of constant columns:", len(constant_cols))

# Drop constant columns
#df.drop(columns=constant_cols, inplace=True)

Constant columns: ['Fwd PSH Flags', 'Fwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init Fwd Win Bytes', 'Fwd Seg Size Min']
Number of constant columns: 10


In [20]:
# Drop constant and unnecessary columns
#df.drop(constant_cols)

In [21]:
print(df.duplicated().sum(), "fully duplicate rows to remove")

88 fully duplicate rows to remove


In [22]:
df.drop_duplicates(inplace=True)
df.reset_index(inplace=True, drop=True)

In [23]:
# Keep Label and Attack for splitting/evaluation; never feed either to the model.
df = df.loc[:, ~df.columns.str.lower().str.startswith(('unnamed:', 'index'))]
feature_cols = [c for c in df if c not in ('Label', 'Attack')]

# Protocol identifiers, DNS query types, and FTP return codes are nominal categories.
categorical_cols = [c for c in col_categorical if c in feature_cols]
for col in categorical_cols:
    df[col] = df[col].astype('string').fillna('missing').astype("string[pyarrow]").str.strip()
numeric_cols = [c for c in feature_cols if c not in categorical_cols]
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
print(f'{len(numeric_cols)} numeric and {len(categorical_cols)} categorical features')

76 numeric and 1 categorical features


In [24]:
print('Categorical:', categorical_cols)
print('Numeric:', numeric_cols)

Categorical: ['Protocol']
Numeric: ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE F

In [25]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)    
print(df.isna().any(axis=1).sum(), "rows with at least one NaN to remove")
# df.dropna(inplace=True)

0 rows with at least one NaN to remove


## 5. Optionally reserve an unseen attack

In [26]:
# Remove incomplete targets and exact repeated rows before any split.
if HOLD_OUT_ATTACK:
    zero_day = df[df['Attack'].eq(HELD_OUT_ATTACK)]
    known = df[df['Attack'].ne(HELD_OUT_ATTACK)]
    print(f'Eligible: {len(known):,} | held-out {HELD_OUT_ATTACK}: {len(zero_day):,}')
else:
    known = df
    zero_day = df.iloc[:0]  # Keep the same schema; no zero-day file is saved.
    print(f'Holdout disabled: all {len(known):,} rows enter the normal split.')


Holdout disabled: all 339,148 rows enter the normal split.


## 6. Split eligible traffic (70% / 15% / 15%)

In [27]:
# Stratify eligible traffic by Attack; exclude the held-out attack only when enabled.
train, remainder = train_test_split(
    known, test_size=0.30, stratify=known['Attack'], random_state=SEED
)
val, test = train_test_split(
    remainder, test_size=0.50, stratify=remainder['Attack'], random_state=SEED
)
# Drop intermediate frames; retain only the four clean, untransformed splits.
del df, known, remainder
print('train / val / test / zero-day:', *(len(x) for x in (train, val, test, zero_day)))

train / val / test / zero-day: 237403 50872 50873 0


## Outliers

In [28]:
# Only numerical measurements—not categories or flag indicators.
outlier_cols = [
    c for c in numeric_cols
    if c not in col_to_one_hot
    and not c.startswith((
        'TCP_FLAGS_',
        'CLIENT_TCP_FLAGS_',
        'SERVER_TCP_FLAGS_',
    ))
]

rows = []

for col in outlier_cols:
    s = train[col]
    finite = s[np.isfinite(s)]

    q1, median, q3, p99, p999 = finite.quantile(
        [0.25, 0.50, 0.75, 0.99, 0.999]
    )
    iqr = q3 - q1

    # Zero IQR makes this rule uninformative for sparse/zero-heavy features.
    iqr_count = (
        ((finite < q1 - 1.5 * iqr) |
         (finite > q3 + 1.5 * iqr)).sum()
        if iqr > 0 else np.nan
    )

    rows.append({
        'feature': col,
        'missing': s.isna().sum(),
        'infinite': np.isinf(s).sum(),
        'minimum': finite.min(),
        'median': median,
        'p99': p99,
        'p99.9': p999,
        'maximum': finite.max(),
        'IQR': iqr,
        'IQR_flagged': iqr_count,
        'above_p99.9': (finite > p999).sum(),
        'outside_float32': (
            (finite > np.finfo(np.float32).max) |
            (finite < -np.finfo(np.float32).max)
        ).sum(),
        'outside_float64': (
            (finite > np.finfo(np.float64).max) |
            (finite < -np.finfo(np.float64).max)
        ).sum(),
    })

outlier_report = pd.DataFrame(rows).set_index('feature')
with pd.option_context('display.max_rows', None):
    display(outlier_report)

,missing,infinite,minimum,median,p99,p99.9,maximum,IQR,IQR_flagged,above_p99.9,outside_float32,outside_float64
feature,,,,,,,,,,,,
Flow Duration,0,0,1.000000,2.335141e+07,1.199321e+08,1.199947e+08,1.200000e+08,1.743055e+07,7115.0,238,0,0
Total Fwd Packets,0,0,0.000000,4.000000e+00,2.392000e+03,6.686598e+03,4.915200e+04,3.000000e+00,21454.0,238,0,0
Total Backward Packets,0,0,1.000000,1.000000e+00,9.000000e+00,3.425000e+03,2.836800e+04,1.000000e+00,15613.0,237,0,0
Fwd Packets Length Total,0,0,0.000000,1.000000e+02,1.947597e+06,6.115592e+06,1.864583e+08,3.000000e+02,6407.0,238,0,0
Bwd Packets Length Total,0,0,0.000000,1.000000e+02,1.215000e+03,3.192817e+06,1.963032e+08,1.000000e+02,9153.0,238,0,0
Fwd Packet Length Max,0,0,0.000000,1.000000e+02,1.432000e+03,1.467000e+03,6.424000e+04,1.000000e+02,8073.0,204,0,0
Fwd Packet Length Min,0,0,0.000000,0.000000e+00,4.020000e+02,1.124000e+03,1.448000e+03,1.000000e+02,3749.0,167,0,0
Fwd Packet Length Mean,0,0,0.000000,3.333333e+01,8.891989e+02,1.124000e+03,5.126219e+04,1.000000e+02,5866.0,189,0,0
Fwd Packet Length Std,0,0,0.000000,0.000000e+00,3.504841e+02,5.860047e+02,2.356144e+04,3.779645e+01,9575.0,238,0,0


## 8. Fit preprocessing on training data only

This CIC notebook previously had no PCA. `PCA_COMPONENTS = None` preserves its existing numeric outputs. To enable PCA over **all scaled numeric measurement columns**, choose an integer component count (no larger than the number of numeric features and fitting rows). Protocol one-hot columns stay outside PCA.

Set `PCA_FIT_ROWS = IMPUTER_FIT_ROWS` for sampled PCA, `None` for full-training PCA, or another positive cap. Pass these into `make_preprocessor(..., pca_fit_rows=PCA_FIT_ROWS, n_components=PCA_COMPONENTS)`. PCA changes the numeric output columns and requires retraining downstream models.

The CIC imputer and numeric scaler retain their existing sampled fitting behavior, now controlled by `IMPUTER_FIT_ROWS`. One-hot vocabularies include all observed training categories. Learned branches are frozen and assembled with one preview row, avoiding a combined full-training feature matrix. All fitting uses training rows only; transforms/writes remain batched. Full-train PCA mode can still require substantial RAM. Requires scikit-learn >= 1.6.

IterativeImputer uses `skip_complete=True` to skip iterative regressions for complete fitting columns. Other fitting settings and optional numeric PCA remain unchanged.


In [29]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

def category_fit_frame(data, columns):
    """Fit independent category vocabularies without expanding millions of rows."""
    values = {c: pd.Series(data[c]).drop_duplicates().to_numpy() for c in columns}
    longest = max(map(len, values.values()))
    # Repeat real values to align column lengths; do not invent missing categories.
    return pd.DataFrame({c: np.resize(v, longest) for c, v in values.items()})


def fit_pca(data, columns, preprocessing, n_components=1, fit_rows=None,
            random_state=42, verbose=True):
    """fit_rows=None uses all training rows; an integer caps PCA fitting rows."""
    if fit_rows is not None and fit_rows < 1:
        raise ValueError('fit_rows must be None or a positive row count.')
    if fit_rows is None or fit_rows >= len(data):
        fit_data = data[columns]
    else:
        fit_data = data.sample(n=fit_rows, random_state=random_state)[columns]
    if verbose:
        print(f'PCA fit: {len(fit_data):,}/{len(data):,} training rows; '
              f'{len(columns)} inputs -> {n_components} component(s).', flush=True)
    model = make_pipeline(
        *[step for _, step in preprocessing.steps],
        PCA(n_components=n_components, svd_solver='full'),
        verbose=verbose,
    )
    model.fit(fit_data)
    model.fit_rows_ = len(fit_data)
    if verbose:
        retained = model.steps[-1][1].explained_variance_ratio_.sum()
        print(f'PCA fitted; retained variance: {retained:.2%}.', flush=True)
    return model


def make_preprocessor(train, numeric_cols, one_hot, scaler=None,
                      imputer_fit_rows=500_000, pca_fit_rows=None,
                      n_components=None, random_state=42, verbose=True):
    """Optional PCA over scaled numeric measurements; categories remain separate."""
    sample = train.sample(
        n=min(imputer_fit_rows, len(train)), random_state=random_state,
    )[numeric_cols]
    imputer = IterativeImputer(max_iter=5, random_state=random_state,
                               skip_complete=True, verbose=int(verbose))
    numeric_scaler = StandardScaler() if scaler is None else clone(scaler)
    numbers = make_pipeline(imputer, numeric_scaler, verbose=verbose)
    if verbose:
        print(f'Fitting CIC imputer/scaler on {len(sample):,} training rows...', flush=True)
    numbers.fit(sample)
    imputer_rows = len(sample)
    del sample

    # Preserve the original CIC numeric fit cap. PCA has its own independent cap.
    if n_components is not None:
        numbers_branch = fit_pca(
            train, numeric_cols, preprocessing=make_pipeline(FrozenEstimator(numbers)),
            n_components=n_components, fit_rows=pca_fit_rows,
            random_state=random_state, verbose=verbose,
        )
    else:
        numbers_branch = numbers
        if verbose:
            print('CIC PCA disabled (n_components=None).', flush=True)
    categories = OneHotEncoder(handle_unknown='ignore').fit(
        category_fit_frame(train, one_hot)
    )
    result = ColumnTransformer([
        ('numbers', FrozenEstimator(numbers_branch), numeric_cols),
        ('categories', FrozenEstimator(categories), one_hot),
    ], verbose=verbose)
    result.fit(train.iloc[:1])
    result.imputer_ = imputer
    result.numeric_scaler_ = numeric_scaler
    result.imputer_fit_rows_ = imputer_rows
    result.pca_ = numbers_branch.steps[-1][1] if n_components is not None else None
    if verbose:
        print(f'CIC preprocessor ready: {len(result.get_feature_names_out()):,} features.', flush=True)
    return result


preprocessor = make_preprocessor(
    train, numeric_cols, col_to_one_hot,
    scaler=NUMERIC_SCALER, imputer_fit_rows=IMPUTER_FIT_ROWS,
    pca_fit_rows=PCA_FIT_ROWS, n_components=PCA_COMPONENTS,
    random_state=SEED, verbose=PREPROCESSING_VERBOSE,
)


Fitting CIC imputer/scaler on 237,403 training rows...
[IterativeImputer] Completing matrix with shape (237403, 76)
[IterativeImputer] Change: 0.0, scaled tolerance: 917530.88 
[IterativeImputer] Early stopping criterion reached.
[Pipeline] .. (step 1 of 2) Processing iterativeimputer, total=   1.4s
[Pipeline]  (step 2 of 2) Processing quantiletransformer, total=   0.1s
CIC PCA disabled (n_components=None).
[IterativeImputer] Completing matrix with shape (1, 76)
[ColumnTransformer] ....... (1 of 2) Processing numbers, total=   0.0s
[ColumnTransformer] .... (2 of 2) Processing categories, total=   0.0s
CIC preprocessor ready: 79 features.


Save each split below in batches, then release its original frame. To repeat saving or run the optional raw-data benchmarks afterward, rerun the loading/splitting cells first.

## 9. Transform, save, and free each split

In [30]:
import pyarrow as pa

feature_names = preprocessor.get_feature_names_out().tolist()

original_target_schema = pa.Schema.from_pandas(
    train[['Label', 'Attack']], preserve_index=False
)
print(original_target_schema)
schema = pa.schema(
    [
        (
            name,
            pa.uint8() if name.startswith('categories__')
            else pa.float32()
        )
        for name in feature_names
    ]
    + [
        original_target_schema.field('Label'),
        original_target_schema.field('Attack'),
    ]
)

print(f'Compact output schema:\n{schema}')

Label: int8
Attack: large_string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 309
Compact output schema:
numbers__Flow Duration: float
numbers__Total Fwd Packets: float
numbers__Total Backward Packets: float
numbers__Fwd Packets Length Total: float
numbers__Bwd Packets Length Total: float
numbers__Fwd Packet Length Max: float
numbers__Fwd Packet Length Min: float
numbers__Fwd Packet Length Mean: float
numbers__Fwd Packet Length Std: float
numbers__Bwd Packet Length Max: float
numbers__Bwd Packet Length Min: float
numbers__Bwd Packet Length Mean: float
numbers__Bwd Packet Length Std: float
numbers__Flow Bytes/s: float
numbers__Flow Packets/s: float
numbers__Flow IAT Mean: float
numbers__Flow IAT Std: float
numbers__Flow IAT Max: float
numbers__Flow IAT Min: float
numbers__Fwd IAT Total: float
numbers__Fwd IAT Mean: float
numbers__Fwd IAT Std: float
numbers__Fwd IAT Max: float
numbers__Fwd IAT Min: float
numbers__Bwd IAT Total: float


# Output folders are retained for local checkpoint resume.
# Use a new output run rather than deleting existing checkpoints here.


In [31]:
from datetime import datetime
from time import perf_counter
import gc
import joblib
import sklearn
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.sparse import issparse

PROJECT = Path(OUTPATH)
fitted_imputer = preprocessor.imputer_
fitted_numeric_scaler = preprocessor.numeric_scaler_
fitted_pca = preprocessor.pca_
attack_tag = f"holdout_{HELD_OUT_ATTACK}" if HOLD_OUT_ATTACK else "all_attacks"
RUN_NAME = f"{datetime.now():%Y%m%d_%H%M%S}_{attack_tag}_{type(fitted_imputer).__name__}"
OUTPUT_DIR = PROJECT / 'outputs' / DATASET / 'preprocessing' / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 10000  # Reduce this if a transformed batch still uses too much RAM.
print('Saving fitted preprocessor to:', OUTPUT_DIR, flush=True)
joblib.dump(preprocessor, OUTPUT_DIR / 'preprocessor.joblib', compress=0)
print('Fitted preprocessor saved.', flush=True)


# Exact baseline from the same cleaned, unencoded TRAIN rows used for export.
import json
raw_category_rows = []
for column in col_to_one_hot:
    unique_values = train[column].drop_duplicates()
    raw_category_rows.append({
        'Raw feature': column,
        'Raw unique entries': len(unique_values),
        'Raw values (first 20)': unique_values.head(20).astype(str).tolist(),
        'Raw missing rows': int(train[column].isna().sum()),
    })
    del unique_values
raw_category_lookup = {r['Raw feature']: r for r in raw_category_rows}
encoded_category_rows = []
for column in preprocessor.get_feature_names_out():
    if not column.startswith('categories__'):
        continue
    inputs = [c for c in col_to_one_hot if column.startswith(f'categories__{c}_')]
    encoded_category_rows.append({
        'Raw feature(s)': ' + '.join(inputs),
        'Raw unique entries': {c: raw_category_lookup[c]['Raw unique entries'] for c in inputs},
        'Raw values (preview)': {c: raw_category_lookup[c]['Raw values (first 20)'] for c in inputs},
        'Encoding': 'One-hot',
        'Encoded column': column,
    })
(OUTPUT_DIR / 'raw_categorical_summary.json').write_text(json.dumps({
    'training_rows': len(train),
    'sampling_max_per_class': MAX_PER_CLASS,
    'seed': SEED,
    'raw_features': raw_category_rows,
    'encoded_features': encoded_category_rows,
}, indent=2), encoding='utf-8')
del raw_category_rows, raw_category_lookup, encoded_category_rows

# Move frames into the queue; remove the other references so they can be freed.
splits = {'train': train, 'val': val, 'test': test}
if HOLD_OUT_ATTACK:
    splits['zero_day'] = zero_day
del train, val, test, zero_day
counts = []
for name in list(splits):
    rows = splits.pop(name)
    split_started = perf_counter()
    total_batches = (len(rows) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f'Starting {name}: {len(rows):,} rows in {total_batches:,} batches.', flush=True)
    with pq.ParquetWriter(OUTPUT_DIR / f'{name}.parquet', schema, compression='zstd') as writer:
        for start in range(0, len(rows), BATCH_SIZE):
            batch = rows.iloc[start:start + BATCH_SIZE]
            X = preprocessor.transform(batch[feature_cols])
            # # Reduce batch memory before densifying sparse output.
            # # float32 rounding is intentional for compact model inputs.
            # X = X.astype(np.float32, copy=False)
            # Densify this small batch only; never the complete split.
            if issparse(X):
                X = X.toarray()
            prepared = pd.DataFrame(X, columns=feature_names)
            prepared['Label'] = batch['Label'].to_numpy()
            prepared['Attack'] = batch['Attack'].to_numpy()
            table = pa.Table.from_pandas(prepared, schema=schema, preserve_index=False)
            writer.write_table(table)
            batch_number = start // BATCH_SIZE + 1
            if (batch_number == 1 or batch_number == total_batches
                    or batch_number % max(1, PRINT_EVERY_BATCHES) == 0):
                completed = min(start + BATCH_SIZE, len(rows))
                print(f'{name}: batch {batch_number:,}/{total_batches:,}; '
                      f'{completed:,}/{len(rows):,} rows written; '
                      f'{format_duration(perf_counter() - split_started)} elapsed.', flush=True)
            del batch, X, prepared, table
    counts.append(f'- {name}: {len(rows):,} rows')
    print(f'Saved {name}: {len(rows):,} rows in {format_duration(perf_counter() - split_started)}; releasing raw split.', flush=True)
    del rows
    gc.collect()  # Frees Python references; the allocator may retain some reserved RAM.

holdout_note = f"Reserve `{HELD_OUT_ATTACK}` as zero-day data" if HOLD_OUT_ATTACK else "Use all attack classes; no separate zero-day set"
saved_files = '`train.parquet`, `val.parquet`, `test.parquet`' + (', `zero_day.parquet`' if HOLD_OUT_ATTACK else '')
readme = f"""# {DATASET} preprocessing — {RUN_NAME}

Source: `{DATA_DIR / FILES[DATASET]}`
Holdout enabled: `{HOLD_OUT_ATTACK}`. Held-out attack: `{HELD_OUT_ATTACK if HOLD_OUT_ATTACK else "None (all attacks included)"}`. Seed: `{SEED}`. scikit-learn: `{sklearn.__version__}`.

## Sampling and comparison
Before cleaning and splitting, retain up to {MAX_PER_CLASS:,} random rows per Attack class and all rows from smaller classes. Compare the actual sample to the full original using class proportions, within-class numerical KS distance, and categorical total variation. Diagnostics do not use another reference sample. Validation/test reflect the capped class mix.

## Steps followed
1. Inspect data and attack counts; strip column/attack-name whitespace.
2. Remove dataset identifiers, IP addresses, and generated index columns.
3. Encode the configured CIC categories `{col_to_one_hot}` using training-only one-hot vocabularies; replace numeric infinities with NaN.
4. Remove rows missing targets and exact duplicate rows.
5. {holdout_note}; split eligible traffic 70/15/15, stratified by Attack. With holdout disabled, attack classes are eligible for every split; rare classes are not guaranteed to occur in each split.
6. Fit `{fitted_imputer!r}` and `{fitted_numeric_scaler!r}` on {preprocessor.imputer_fit_rows_:,} sampled training rows (cap {IMPUTER_FIT_ROWS:,}). One-hot vocabularies include all observed training categories. Optional numeric PCA: {PCA_COMPONENTS!r} components; requested PCA row cap: {PCA_FIT_ROWS!r}; actual PCA fit rows: {fitted_pca.n_samples_ if fitted_pca is not None else 'disabled'}. PCA variance retained: {fitted_pca.explained_variance_ratio_.sum() if fitted_pca is not None else 'disabled'}. All fitting uses training data only; branches are frozen and assembled with one preview row.
7. Transform each split in batches of at most {BATCH_SIZE:,} rows and write directly to Parquet with Zstandard compression.
8. Release each batch and completed split before continuing.

## Runtime, checkpoints and GPU
All measured durations shown by this notebook use hours/minutes/seconds. Numeric seconds are retained internally in checkpoints. The resampling workflow uses four small shared helpers; optional prepared dtypes/outlier inspections are under Testing.
This notebook caps sampled rows at 100,000 per Attack before the split. Main ADASYN selects classes below 0.5 of the largest TRAIN class and targets 0.5 of that count, with a 3% tolerance around the requested target. Native class names and binary Label remain separate from model features.
Per-policy resampling/ directories hold full uncompressed ADASYN checkpoints, validated using SHA256 of prepared training content, feature order, seed, cap, policy parameters and package versions. GPU backend status is recorded; restored times describe original computation, not reload time. Optional cuML activation runs before sklearn imports, logs actual dispatch/fallback, and may also accelerate other supported estimators. CPU/GPU neighbours can differ at ties or due to numerical arithmetic.
Outputs and checkpoints are saved in OUTPUT_DIR. Save Kaggle outputs before discarding a session.
The optional RUN_THRESHOLD_TESTS block uses main prepared train/validation inputs for an unresampled baseline and paired selection/target policies 0.25, 0.5 and 0.75. Each uses the same 32-unit MLP/Adam/30-epoch settings. Test data is never used by those comparisons. Results/models/confusion matrices are saved below threshold_tests/. Completed-result reuse and original compute timings are labelled explicitly.

## Saved files
`raw_categorical_summary.json` records raw training categorical domains and encoded-column mappings. The imbalance cells save `adasyn_generated_preview.parquet` with genuine generated rows and Tomek retention flags; its metadata fields are not model inputs.
{saved_files} contain TRANSFORMED numeric feature columns plus `Label` and `Attack`.
Features are already imputed, encoded, and scaled: do not transform them again.
`preprocessor.joblib` is for new RAW data only. Feature order is `preprocessor.get_feature_names_out()`.
Storage types: scaled numerical features or optional numeric PCA outputs use float32; one-hot features use uint8. Protocol one-hot outputs remain outside PCA. IterativeImputer skips iterative regressions for complete fitting columns.
Label and Attack retain their original Arrow types. The same schema is used for every batch and split.
Float32 reduces precision compared with float64.
These files differ from older exports of cleaned, untransformed features.

{chr(10).join(counts)}

## Resampling and diagnostics
After preprocessing/export, ADASYN operates on prepared training data, then Tomek Links cleans the original majority Attack class. Validation/test are not resampled. Synthetic feature coordinates remain floating point. The resampling cells report start/end times, shapes, matrix size and row deltas; no periodic status messages are printed. Tomek Links, result tables and Plotly are separate cells. Enable RUN_CATEGORY_DIAGNOSTICS in Testing to compare exact raw/encoded/resampled categorical domains and inspect the saved genuine generated-row preview.

## Read prepared batches
```python
import pyarrow.parquet as pq
for record_batch in pq.ParquetFile('train.parquet').iter_batches(batch_size=10000):
    batch = record_batch.to_pandas()
    X = batch.drop(columns=['Label', 'Attack'])
    y = batch['Label']
    # Consume this batch directly; do not apply preprocessor.transform(X).
```

Initial loading/splitting and imputer fitting still require RAM. Full-train PCA can require substantial RAM; set PCA_FIT_ROWS to a smaller cap. To reload preprocessor.joblib, use scikit-learn >= 1.6; no custom transformer class is required. Saving releases the original split frames; rerun loading/splitting before using raw-data benchmarks or repeating the save step.
Row splits and exact deduplication do not guarantee session isolation or isolation of identical predictors with conflicting labels.
"""
(OUTPUT_DIR / 'README.md').write_text(readme, encoding='utf-8')
print('Prepared datasets saved to:', OUTPUT_DIR)

Saving fitted preprocessor to: /kaggle/working/project/outputs/CIC-BoT-IoT/preprocessing/20261116_112425_all_attacks_IterativeImputer
Fitted preprocessor saved.
Starting train: 237,403 rows in 24 batches.
[IterativeImputer] Completing matrix with shape (10000, 76)
train: batch 1/24; 10,000/237,403 rows written; 00h 00m 00.3s elapsed.
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
[IterativeImputer] Completing matrix with shape (10000, 76)
train: batch 10/24; 100,000/237,403 rows written; 00h 00m 02.7s elapsed.
[IterativeImputer] Completing matrix with shape (10

## 7. Class Imbalance

Main order: clean/split → fit imputation/encoding/scaling on train → transform/save → ADASYN on train → Tomek Links → model training.

Load the processed `train.parquet`, resample its `Attack` classes with ADASYN, then clean the original majority Attack class with Tomek Links. Exclude both targets from features. Restore binary `Label` from the original Attack-to-Label mapping when saving. Validation and test are not resampled here. Synthetic one-hot coordinates can be fractional and remain floating point.
Run the cells below in order: settings → shared loading/preview tools → checkpoint tools → restore checkpoint → load training if needed → ADASYN → save checkpoint → Tomek → tables → plot → save balanced train. Only four reused helpers remain here. All measured durations are displayed as `00h 00m 00.0s`; numeric seconds remain internal to checkpoint calculations.


In [32]:
HANDEL_IMBALANCE = True
SELECT_BELOW_RATIO = 0.50
TARGET_RATIO = 0.50
ADASYN_BALANCE_TOLERANCE = 0.03  # Skip negligible top-ups around the requested target.
REUSE_ADASYN_CHECKPOINT = True
if not 0 < SELECT_BELOW_RATIO <= TARGET_RATIO <= 1:
    raise ValueError('Require 0 < selection ratio <= target ratio <= 1.')
# Fresh-session resume: set an EXISTING output run here; do not rerun main export.
RESUME_OUTPUT_DIR = None  # e.g. '/kaggle/working/project/outputs/NF_TON_IOT/preprocessing/<run>'
if RESUME_OUTPUT_DIR is not None:
    OUTPUT_DIR = Path(RESUME_OUTPUT_DIR)


### Shared loading and sample-preview tools

These two functions are reused by the main run and the three threshold comparisons. The batched loader retains the memory guard and checks the Attack-to-Label mapping; the preview contains genuine appended ADASYN rows.


In [33]:
from pathlib import Path
from time import perf_counter
import gc
import json
import math
import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import plotly.express as px
import psutil
import sklearn
import imblearn
from imblearn.over_sampling import ADASYN
from imblearn.under_sampling import TomekLinks

# Load a single dense matrix in batches; keep fractional synthetic coordinates.
def read_prepared_matrix(path, feature_names=None):
    parquet = pq.ParquetFile(path)
    schema = parquet.schema_arrow
    names = feature_names or [c for c in schema.names if c not in ('Label', 'Attack')]
    dtype = np.result_type(np.float32, *[
        np.dtype(schema.field(c).type.to_pandas_dtype()) for c in names
    ])
    rows = parquet.metadata.num_rows
    required = rows * (len(names) * dtype.itemsize + np.dtype(object).itemsize)
    available = psutil.virtual_memory().available
    print(f'Loading {rows:,} rows × {len(names):,} features; dtype={dtype}; '
          f'working arrays ≈ {required / 2**30:.2f} GiB; '
          f'available RAM={available / 2**30:.2f} GiB', flush=True)
    if required > available * 0.5:
        raise MemoryError('Working arrays exceed half the available RAM. Reduce training rows.')
    X = np.empty((rows, len(names)), dtype=dtype)
    y = np.empty(rows, dtype=object)
    target_parts = []
    start = 0
    for record in parquet.iter_batches(batch_size=20_000, columns=names + ['Attack', 'Label'], use_threads=False):
        frame = record.to_pandas()
        stop = start + len(frame)
        for j, column in enumerate(names):
            X[start:stop, j] = frame[column].to_numpy(copy=False)
        y[start:stop] = frame['Attack'].to_numpy(copy=False)
        target_parts.append(frame[['Attack', 'Label']].drop_duplicates())
        start = stop
    pairs = pd.concat(target_parts, ignore_index=True).drop_duplicates()
    if pairs['Attack'].duplicated().any():
        raise ValueError('An Attack class has multiple Label values.')
    metadata = {
        'original_columns': schema.names,
        'target_dtypes': pairs.dtypes,
        'attack_to_label': pairs.set_index('Attack')['Label'],
    }
    return X, y, names, metadata

def generated_row_preview(X, y, input_rows, targets, names, label_map):
    positions = []
    for attack in targets:
        positions.extend((input_rows + np.flatnonzero(y[input_rows:] == attack)[:5]).tolist())
    columns = [c for c in names if c.startswith('categories__')]
    indices = [names.index(c) for c in columns]
    preview = pd.DataFrame(X[np.ix_(positions, indices)], columns=columns)
    preview.insert(0, 'ADASYN row', positions)
    preview['Attack'] = y[positions]
    preview['Label'] = preview['Attack'].map(label_map)
    preview['Kept after Tomek'] = pd.Series([None] * len(preview), dtype='boolean')
    return preview


### Checkpoint tools

Two reused functions calculate the prepared-file fingerprint and save a complete checkpoint atomically. They preserve compatibility with existing ADASYN checkpoints.


In [34]:
import hashlib

_file_signatures = {}

def prepared_signature(path):
    """Hash once per changed file; content identity survives copying the prepared output run."""
    path = Path(path)
    stat = path.stat()
    key = (str(path.resolve()), stat.st_size, stat.st_mtime_ns)
    if key not in _file_signatures:
        digest = hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
                digest.update(block)
        _file_signatures[key] = digest.hexdigest()
    return _file_signatures[key]

def atomic_checkpoint(path, state, settings):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.tmp')
    joblib.dump({'settings': settings, 'state': state}, temporary, compress=0)
    temporary.replace(path)
    print('Checkpoint saved:', path, flush=True)


### 1. Prepare settings and restore a matching ADASYN checkpoint

For fresh-session resume, run Setup (including format_duration), dataset/sampling constants, imbalance settings and the two utility cells. Set RESUME_OUTPUT_DIR to an existing prepared run, then run this cell. A matching checkpoint skips training-array loading and ADASYN. All source/settings compatibility checks remain; mismatches require REUSE_ADASYN_CHECKPOINT=False to recompute.

This cell also prepares the identity needed by the optional threshold/model tests, including when main resampling is disabled. Keep the complete prepared output run when moving between sessions.


In [35]:
# For a fresh session, set OUTPUT_DIR to the existing prepared run before this cell.
# Run Setup, imbalance settings, the two utility cells, and this restore cell first.
# This identity is also used by the optional prepared-data threshold tests.
current_policy_settings = {
    'format': 1, 'train_sha256': prepared_signature(OUTPUT_DIR / 'train.parquet'),
    'features': [c for c in pq.read_schema(OUTPUT_DIR / 'train.parquet').names
                 if c not in ('Label', 'Attack')],
    'dataset': DATASET, 'seed': SEED, 'max_per_class': MAX_PER_CLASS,
    'selection': SELECT_BELOW_RATIO, 'target': TARGET_RATIO,
    'tolerance': ADASYN_BALANCE_TOLERANCE, 'max_neighbors': 5,
    'sklearn': sklearn.__version__, 'imblearn': imblearn.__version__,
}
ADASYN_CACHE_LOADED = False
if HANDEL_IMBALANCE:
    RESAMPLING_DIR = OUTPUT_DIR / 'resampling' / f'select_{SELECT_BELOW_RATIO:g}_target_{TARGET_RATIO:g}'
    RESAMPLING_DIR.mkdir(parents=True, exist_ok=True)
    ADASYN_CHECKPOINT = RESAMPLING_DIR / 'adasyn_checkpoint.joblib'
    if REUSE_ADASYN_CHECKPOINT and ADASYN_CHECKPOINT.exists():
        checkpoint = joblib.load(ADASYN_CHECKPOINT, mmap_mode='r')
        if checkpoint['settings'] != current_policy_settings:
            raise ValueError('Checkpoint settings/data differ. Set REUSE_ADASYN_CHECKPOINT=False to recompute.')
        restored_state = checkpoint['state']
        globals().update(restored_state)
        del restored_state, checkpoint
        ADASYN_CACHE_LOADED = True
        print(f'Restored ADASYN checkpoint: X={X_adasyn.shape}, y={y_adasyn.shape}', flush=True)
        print('Original ADASYN compute time:', format_duration(adasyn_seconds), '| backend:', adasyn_backend)
        generated_preview.to_parquet(OUTPUT_DIR / 'adasyn_generated_preview.parquet', index=False)


In [36]:
if HANDEL_IMBALANCE and not ADASYN_CACHE_LOADED:
    X_train, y_train, resampling_features, train_metadata = read_prepared_matrix(
        OUTPUT_DIR / 'train.parquet',
    )
    original_columns = train_metadata['original_columns']
    target_dtypes = train_metadata['target_dtypes']
    attack_to_label = train_metadata['attack_to_label']
    before_counts = pd.Series(y_train).value_counts().sort_index()
    majority_attack = before_counts.idxmax()
    display(before_counts.rename_axis('Attack').rename('rows').to_frame())
    del train_metadata


Loading 237,403 rows × 79 features; dtype=float32; working arrays ≈ 0.07 GiB; available RAM=29.66 GiB


,rows
Attack,
Benign,26848
DDoS,70000
DoS,70000
Reconnaissance,69994
Theft,561


### 2. Select minority classes and run ADASYN

Selection and requested counts are calculated directly in the next code cell. With the default 0.5/0.5 policy, select classes below half the largest TRAIN class and request half its count; skip additions within 3% of that requested count. Unselected classes stay in the reference data. ADASYN failures remain visible without substituting another sampler.

Start/end output, post-ADASYN shape, added rows and matrix size remain visible. GPU dispatch is reported by startup/cuML logs. Cached times describe the original computation.


### ADASYN — timing, shape and generated-row preview


In [37]:
if HANDEL_IMBALANCE and not ADASYN_CACHE_LOADED:
    input_train_rows = len(y_train)
    print(f'Before ADASYN: X={X_train.shape}, y={y_train.shape}', flush=True)
    adasyn_seconds = 0.0
    adasyn_skipped = False
    requested_count = math.ceil(TARGET_RATIO * int(before_counts.max()))
    sampling_targets = {
        attack: requested_count for attack, count in before_counts.items()
        if count < SELECT_BELOW_RATIO * before_counts.max()
        and count < (1 - ADASYN_BALANCE_TOLERANCE) * requested_count
    }
    print('Selected class targets:', sampling_targets, flush=True)

    if sampling_targets:
        smallest_target_class = min(
            int(before_counts[attack]) for attack in sampling_targets
        )
        if smallest_target_class < 2:
            raise ValueError(
                'ADASYN needs at least two examples in each targeted Attack class.'
            )

        adasyn = ADASYN(
            sampling_strategy=sampling_targets,
            n_neighbors=min(5, smallest_target_class - 1),
            random_state=SEED,
        )
        adasyn_started = perf_counter()
        print('ADASYN: starting...', flush=True)
        X_adasyn, y_adasyn = adasyn.fit_resample(X_train, y_train)
        adasyn_seconds = perf_counter() - adasyn_started
    else:
        adasyn_skipped = True
        print('No classes require the selected partial-balancing policy; skipping ADASYN.', flush=True)
        X_adasyn, y_adasyn = X_train, y_train

    adasyn_rows = len(y_adasyn)
    print(f'After ADASYN: X={X_adasyn.shape}, y={y_adasyn.shape}', flush=True)
    print(f'ADASYN time: {format_duration(adasyn_seconds)} | '
          f'Rows added: {adasyn_rows - input_train_rows:,}', flush=True)
    print(f'Post-ADASYN feature matrix: {X_adasyn.nbytes / 2**30:.2f} GiB | '
          f'dtype={X_adasyn.dtype}', flush=True)

    adasyn_counts = pd.Series(y_adasyn).value_counts().sort_index()
    print('After ADASYN (Attack):')
    display(pd.DataFrame({
        'Before': before_counts,
        'After ADASYN': adasyn_counts,
    }).fillna(0).astype(int).rename_axis('Attack'))

    adasyn_backend = GPU_STATUS if sampling_targets else 'skipped'
    generated_preview = generated_row_preview(
        X_adasyn, y_adasyn, input_train_rows, sampling_targets,
        resampling_features, attack_to_label,
    )
    generated_preview.to_parquet(OUTPUT_DIR / 'adasyn_generated_preview.parquet', index=False)

    del X_train, y_train
    gc.collect()

Before ADASYN: X=(237403, 79), y=(237403,)
Selected class targets: {'Benign': 35000, 'Theft': 35000}
ADASYN: starting...
After ADASYN: X=(280271, 79), y=(280271,)
ADASYN time: 00h 00m 35.0s | Rows added: 42,868
Post-ADASYN feature matrix: 0.08 GiB | dtype=float32
After ADASYN (Attack):


,Before,After ADASYN
Attack,,
Benign,26848,35252
DDoS,70000,70000
DoS,70000,70000
Reconnaissance,69994,69994
Theft,561,35025


### 3. Save the full ADASYN checkpoint

Atomic local save. This checkpoint contains features, targets, counts and preview metadata, retaining the numerical matrix dtype. An interrupted temporary write is not accepted as a completed checkpoint.


In [38]:
if HANDEL_IMBALANCE and not ADASYN_CACHE_LOADED:
    cache_variables = [
        'X_adasyn', 'y_adasyn', 'resampling_features', 'original_columns',
        'target_dtypes', 'attack_to_label', 'before_counts', 'adasyn_counts',
        'majority_attack', 'input_train_rows', 'adasyn_rows', 'adasyn_seconds',
        'adasyn_skipped', 'generated_preview', 'sampling_targets', 'adasyn_backend',
    ]
    cache_state = {name: globals()[name] for name in cache_variables}
    atomic_checkpoint(ADASYN_CHECKPOINT, cache_state, current_policy_settings)
    del cache_state
if HANDEL_IMBALANCE:
    settings_path = RESAMPLING_DIR / 'policy_settings.json'
    settings_path.write_text(json.dumps(current_policy_settings, indent=2))


Checkpoint saved: /kaggle/working/project/outputs/CIC-BoT-IoT/preprocessing/20261116_112425_all_attacks_IterativeImputer/resampling/select_0.5_target_0.5/adasyn_checkpoint.joblib


### 4. Tomek Links — timing and output shape


In [39]:
if HANDEL_IMBALANCE:
    print(f'Before Tomek Links: X={X_adasyn.shape}, y={y_adasyn.shape}', flush=True)
    # Preserve the original policy: clean only the original majority class.
    tomek = TomekLinks(
        sampling_strategy=[majority_attack],
        n_jobs=-1,
    )
    tomek_started = perf_counter()
    print('Tomek Links: starting...', flush=True)
    X_train_balanced, y_train_balanced = tomek.fit_resample(X_adasyn, y_adasyn)
    tomek_seconds = perf_counter() - tomek_started
    tomek_removed_rows = len(y_adasyn) - len(y_train_balanced)
    print(f'After Tomek Links: X={X_train_balanced.shape}, '
          f'y={y_train_balanced.shape}', flush=True)
    print(f'Tomek Links time: {format_duration(tomek_seconds)} | '
          f'Rows removed: {tomek_removed_rows:,}', flush=True)
    after_counts = pd.Series(y_train_balanced).value_counts()

    # Record which preview rows survived, using Tomek's actual selected indices.
    generated_preview['Kept after Tomek'] = np.isin(
        generated_preview['ADASYN row'].to_numpy(), tomek.sample_indices_,
    )
    generated_preview.to_parquet(
        OUTPUT_DIR / 'adasyn_generated_preview.parquet', index=False,
    )
    del generated_preview

    # Free the expanded input before plotting or exporting the balanced output.
    del X_adasyn, y_adasyn
    gc.collect()


Before Tomek Links: X=(280271, 79), y=(280271,)
Tomek Links: starting...
After Tomek Links: X=(257409, 79), y=(257409,)
Tomek Links time: 00h 06m 12.9s | Rows removed: 22,862


### 5. Resampling results and timings


In [40]:
if HANDEL_IMBALANCE:
    distribution = pd.DataFrame({
        'Before': before_counts,
        'After ADASYN': adasyn_counts,
        'After Tomek Links': after_counts,
    }).fillna(0).astype(int).rename_axis('Attack')

    distribution['Before (%)'] = (
        100 * distribution['Before'] / distribution['Before'].sum()
    ).round(2)
    distribution['After (%)'] = (
        100 * distribution['After Tomek Links']
        / distribution['After Tomek Links'].sum()
    ).round(2)
    display(distribution)

    resampling_summary = pd.DataFrame([
        {'Stage': 'ADASYN', 'Status': 'cached' if ADASYN_CACHE_LOADED else ('skipped' if adasyn_skipped else 'finished'),
         'Input rows': input_train_rows, 'Output rows': adasyn_rows,
         'Features': len(resampling_features), 'Time': format_duration(adasyn_seconds)},
        {'Stage': 'Tomek Links', 'Status': 'finished',
         'Input rows': adasyn_rows, 'Output rows': adasyn_rows - tomek_removed_rows,
         'Features': len(resampling_features), 'Time': format_duration(tomek_seconds)},
    ]).set_index('Stage')
    display(resampling_summary)
    distribution_path = RESAMPLING_DIR / 'class_distribution.csv'
    distribution.to_csv(distribution_path)
    timing_path = RESAMPLING_DIR / 'timings.csv'
    resampling_summary.to_csv(timing_path)


,Before,After ADASYN,After Tomek Links,Before (%),After (%)
Attack,,,,,
Benign,26848,35252,35252,11.31,13.69
DDoS,70000,70000,47138,29.49,18.31
DoS,70000,70000,70000,29.49,27.19
Reconnaissance,69994,69994,69994,29.48,27.19
Theft,561,35025,35025,0.24,13.61


,Status,Input rows,Output rows,Features,Time
Stage,,,,,
ADASYN,finished,237403,280271,79,00h 00m 35.0s
Tomek Links,finished,280271,257409,79,00h 06m 12.9s


### 6. Plot class distribution

Run this cell again to redraw the chart without rerunning Tomek Links.


In [41]:
if HANDEL_IMBALANCE:
    # Plot aggregated counts instead of passing millions of rows to Plotly.
    plot_counts = distribution[
        ['Before', 'After ADASYN', 'After Tomek Links']
    ].reset_index().melt(
        id_vars='Attack', var_name='Stage', value_name='Rows'
    )
    plot_counts['Attack'] = plot_counts['Attack'].astype(str)
    fig = px.histogram(
        plot_counts,
        x='Attack',
        y='Rows',
        color='Stage',
        histfunc='sum',
        barmode='group',
        title='Attack class distribution before and after resampling',
        labels={'Rows': 'Number of samples'},
        category_orders={'Attack': distribution.index.astype(str).tolist()},
    )
    fig.update_layout(yaxis_title='Number of samples', xaxis_tickangle=-45)
    plot_path = RESAMPLING_DIR / 'class_distribution.html'
    fig.write_html(plot_path, include_plotlyjs='cdn')
    fig.show()



### 7. Save the balanced training data


In [42]:
if HANDEL_IMBALANCE:
    balanced_train = pd.DataFrame(
        X_train_balanced,
        columns=resampling_features,
    )
    balanced_train['Attack'] = pd.Series(y_train_balanced).astype(
        target_dtypes['Attack']
    )
    balanced_train['Label'] = balanced_train['Attack'].map(
        attack_to_label
    ).astype(target_dtypes['Label'])
    balanced_train = balanced_train[original_columns]

    balanced_path = OUTPUT_DIR / 'train_adasyn_tomek.parquet'
    balanced_train.to_parquet(
        balanced_path,
        index=False,
        compression='zstd',
    )
    print('Saved:', balanced_path)

    del balanced_train, X_train_balanced, y_train_balanced
    gc.collect()

Saved: /kaggle/working/project/outputs/CIC-BoT-IoT/preprocessing/20261116_112425_all_attacks_IterativeImputer/train_adasyn_tomek.parquet


Train using `train_adasyn_tomek.parquet`, with the existing `val.parquet` and `test.parquet`. The saved training file contains both `Attack` and `Label`; use `Attack` as the multiclass target. Exclude both targets from the model inputs. These features are already preprocessed—do not transform them again.

The table and Plotly histograms above show counts before resampling, after ADASYN, and after Tomek Links. Percentages compare the original and final training distributions.

## Github

# Testing

The categorical diagnostic below reads the main saved output run. Run the platform/path/dataset configuration and set `OUTPUT_DIR` to that run, or continue after the main export/resampling cells. No raw-data benchmark or refitting is required. The independent flag controls the test; missing diagnostic snapshots produce prerequisite guidance.


The prepared-data baseline/threshold tests run independently of legacy raw-data benchmarks. Run Setup (including format_duration), dataset/sampling constants, imbalance settings, both utility cells and the checkpoint-restore cell, with OUTPUT_DIR pointing to the completed prepared run. The tests read train.parquet and val.parquet only; test/zero-day data is reserved.


In [43]:
# Enable either or both groups, then run the loading cell immediately below.
RUN_TESTING = True
RUN_CATEGORY_DIAGNOSTICS = True  # Main raw -> encoded -> resampled categorical audit.

RUN_THRESHOLD_TESTS = True  # Baseline plus paired selection/target ratios 0.25/0.5/0.75.


### Categorical unique values and generated samples

Enable `RUN_CATEGORY_DIAGNOSTICS` and use `OUTPUT_DIR` from the main run. This cell reads main saved outputs; it does not refit preprocessing or rerun ADASYN/Tomek.

- Before processing: exact unique counts (including missing) and up to 20 examples from the same cleaned, unencoded training `Protocol` column.
- After processing: exact unique counts and value previews for each `Protocol` one-hot output in `train.parquet`.
- After resampling: exact unique counts, new values and non-binary indicator entries in `train_adasyn_tomek.parquet`, after ADASYN and Tomek Links. `MAX_UNIQUE_EXAMPLES = 12` limits displayed encoded examples; counts use every row. One Parquet column is read at a time to limit memory; full scans can take time.
- Generated samples: up to five genuine appended ADASYN rows per targeted Attack, with encoded coordinates and whether Tomek retained each row. A small preview may miss fractional values found by the full final-domain comparison.

One-hot indicators normally contain 0/1; ADASYN can interpolate fractional coordinates. The diagnostic displays values without rounding or modifying data. Optional CIC PCA transforms only numeric measurements; `Protocol` one-hot outputs stay separate and remain the focus of this test. This compares value domains, not category frequencies or semantic validity.

Run the updated main export and resampling cells to create `raw_categorical_summary.json` and `adasyn_generated_preview.parquet`. Older runs do not contain these snapshots. If raw frames/matrices were released, recreate them through the usual loading/splitting/fitting/export flow before resampling. Arbitrary final rows are never labelled as generated.


In [44]:
if RUN_CATEGORY_DIAGNOSTICS:
    import json
    import numpy as np
    import pandas as pd
    import pyarrow as pa
    import pyarrow.compute as pc
    import pyarrow.parquet as pq

    # OUTPUT_DIR must point to the same run for all three diagnostic files.
    raw_summary_path = OUTPUT_DIR / 'raw_categorical_summary.json'
    processed_path = OUTPUT_DIR / 'train.parquet'
    resampled_path = OUTPUT_DIR / 'train_adasyn_tomek.parquet'
    preview_path = OUTPUT_DIR / 'adasyn_generated_preview.parquet'
    MAX_UNIQUE_EXAMPLES = 12

    if not all(p.exists() for p in [raw_summary_path, processed_path,
                                    resampled_path, preview_path]):
        print('Run the updated main export, ADASYN and Tomek Links cells first. '
              'They create the categorical baseline and generated-row preview '
              'inside OUTPUT_DIR. This test does not rerun resampling.')
    else:
        category_baseline = json.loads(raw_summary_path.read_text())
        raw_category_uniques = pd.DataFrame(category_baseline['raw_features'])
        print('Raw categorical values from the SAME training split before processing:')
        display(raw_category_uniques)
        print('Rows compared: raw/processed training = '
              f"{category_baseline['training_rows']:,}; final resampled training = "
              f'{pq.ParquetFile(resampled_path).metadata.num_rows:,}')

        # Read one encoded column at a time, not either complete feature matrix.
        comparison_rows = []
        for mapping in category_baseline['encoded_features']:
            column = mapping['Encoded column']
            before_unique = pc.unique(pq.read_table(
                processed_path, columns=[column]
            ).column(0).cast(pa.float64()))
            after_unique = pc.unique(pq.read_table(
                resampled_path, columns=[column]
            ).column(0).cast(pa.float64()))
            new_unique = pc.filter(
                after_unique,
                pc.invert(pc.is_in(after_unique, value_set=before_unique)),
            )
            non_binary = int(pc.sum(pc.and_(
                pc.not_equal(after_unique, 0),
                pc.not_equal(after_unique, 1),
            )).as_py() or 0)
            comparison_rows.append({
                **mapping,
                'Unique after processing': len(before_unique),
                'Processed values (preview)': before_unique.slice(0, MAX_UNIQUE_EXAMPLES).to_pylist(),
                'Unique after ADASYN + Tomek': len(after_unique),
                'Resampled values (preview)': after_unique.slice(0, MAX_UNIQUE_EXAMPLES).to_pylist(),
                'New unique entries': len(new_unique),
                'New values (preview)': new_unique.slice(0, MAX_UNIQUE_EXAMPLES).to_pylist(),
                'Non-binary unique entries (final)': non_binary,
            })
            del before_unique, after_unique, new_unique

        category_unique_comparison = pd.DataFrame(comparison_rows)
        print('Before processing -> encoded training -> final resampled training:')
        display(category_unique_comparison)
        print('Indicator columns containing values other than 0 and 1 after resampling:')
        display(category_unique_comparison.loc[
            category_unique_comparison['Non-binary unique entries (final)'].fillna(0).gt(0)
        ].head(30))

        generated_samples = pd.read_parquet(preview_path)
        if generated_samples.empty:
            print('ADASYN generated no rows in this run; the generated preview is empty.')
        else:
            encoded_columns = [m['Encoded column'] for m in category_baseline['encoded_features']]
            indicator_columns = encoded_columns
            generated_encoded_values = generated_samples.melt(
                id_vars=['ADASYN row', 'Attack', 'Label', 'Kept after Tomek'],
                value_vars=indicator_columns,
                var_name='Encoded column', value_name='Generated value',
            )
            generated_fractional_examples = generated_encoded_values.loc[
                ~generated_encoded_values['Generated value'].isin([0, 1])
            ]
            focus_columns = list(dict.fromkeys(
                generated_fractional_examples['Encoded column'].tolist()
                + encoded_columns
            ))[:12]
            print('Genuine ADASYN-generated rows (up to 5 per oversampled Attack):')
            display(generated_samples[
                ['ADASYN row', 'Attack', 'Label', 'Kept after Tomek'] + focus_columns
            ].head(20))
            print('Generated non-binary indicator coordinates (first 40):')
            display(generated_fractional_examples.head(40))
            if generated_fractional_examples.empty:
                print('No fractional indicator values occurred in this small preview. '
                      'Use the full-data unique comparison above for all final rows.')


Raw categorical values from the SAME training split before processing:


,Raw feature,Raw unique entries,Raw values (first 20),Raw missing rows
0,Protocol,3,"[6, 17, 0]",0


Rows compared: raw/processed training = 237,403; final resampled training = 257,409
Before processing -> encoded training -> final resampled training:


,Raw feature(s),Raw unique entries,Raw values (preview),Encoding,Encoded column,Unique after processing,Processed values (preview),Unique after ADASYN + Tomek,Resampled values (preview),New unique entries,New values (preview),Non-binary unique entries (final)
0,Protocol,{'Protocol': 3},"{'Protocol': ['6', '17', '0']}",One-hot,categories__Protocol_0,2,"[0.0, 1.0]",846,"[0.0, 1.0, 0.5751714706420898, 0.38043177127838135, 0.31853988766670227, 0.7463843822479248, 0.438522070646286, 0.8140663504600525, 0.3169816732406616, 0.19805367290973663, 0.27318134903907776, 0.7008571624755859]",844,"[0.5751714706420898, 0.38043177127838135, 0.31853988766670227, 0.7463843822479248, 0.438522070646286, 0.8140663504600525, 0.3169816732406616, 0.19805367290973663, 0.27318134903907776, 0.7008571624755859, 0.09506423026323318, 0.0987200140953064]",844
1,Protocol,{'Protocol': 3},"{'Protocol': ['6', '17', '0']}",One-hot,categories__Protocol_17,2,"[0.0, 1.0]",11,"[0.0, 1.0, 0.42482852935791016, 0.6195682287216187, 0.6814601421356201, 0.2536156177520752, 0.5614779591560364, 0.18593363463878632, 0.6830183267593384, 0.8019463419914246, 0.7268186211585999]",9,"[0.42482852935791016, 0.6195682287216187, 0.6814601421356201, 0.2536156177520752, 0.5614779591560364, 0.18593363463878632, 0.6830183267593384, 0.8019463419914246, 0.7268186211585999]",9
2,Protocol,{'Protocol': 3},"{'Protocol': ['6', '17', '0']}",One-hot,categories__Protocol_6,2,"[1.0, 0.0]",837,"[1.0, 0.0, 0.29914286732673645, 0.9049357771873474, 0.9012799859046936, 0.3619024157524109, 0.941291332244873, 0.07709825038909912, 0.5215959548950195, 0.852748453617096, 0.08220548182725906, 0.35256651043891907]",835,"[0.29914286732673645, 0.9049357771873474, 0.9012799859046936, 0.3619024157524109, 0.941291332244873, 0.07709825038909912, 0.5215959548950195, 0.852748453617096, 0.08220548182725906, 0.35256651043891907, 0.11538472026586533, 0.09900753945112228]",835


Indicator columns containing values other than 0 and 1 after resampling:


,Raw feature(s),Raw unique entries,Raw values (preview),Encoding,Encoded column,Unique after processing,Processed values (preview),Unique after ADASYN + Tomek,Resampled values (preview),New unique entries,New values (preview),Non-binary unique entries (final)
0,Protocol,{'Protocol': 3},"{'Protocol': ['6', '17', '0']}",One-hot,categories__Protocol_0,2,"[0.0, 1.0]",846,"[0.0, 1.0, 0.5751714706420898, 0.38043177127838135, 0.31853988766670227, 0.7463843822479248, 0.438522070646286, 0.8140663504600525, 0.3169816732406616, 0.19805367290973663, 0.27318134903907776, 0.7008571624755859]",844,"[0.5751714706420898, 0.38043177127838135, 0.31853988766670227, 0.7463843822479248, 0.438522070646286, 0.8140663504600525, 0.3169816732406616, 0.19805367290973663, 0.27318134903907776, 0.7008571624755859, 0.09506423026323318, 0.0987200140953064]",844
1,Protocol,{'Protocol': 3},"{'Protocol': ['6', '17', '0']}",One-hot,categories__Protocol_17,2,"[0.0, 1.0]",11,"[0.0, 1.0, 0.42482852935791016, 0.6195682287216187, 0.6814601421356201, 0.2536156177520752, 0.5614779591560364, 0.18593363463878632, 0.6830183267593384, 0.8019463419914246, 0.7268186211585999]",9,"[0.42482852935791016, 0.6195682287216187, 0.6814601421356201, 0.2536156177520752, 0.5614779591560364, 0.18593363463878632, 0.6830183267593384, 0.8019463419914246, 0.7268186211585999]",9
2,Protocol,{'Protocol': 3},"{'Protocol': ['6', '17', '0']}",One-hot,categories__Protocol_6,2,"[1.0, 0.0]",837,"[1.0, 0.0, 0.29914286732673645, 0.9049357771873474, 0.9012799859046936, 0.3619024157524109, 0.941291332244873, 0.07709825038909912, 0.5215959548950195, 0.852748453617096, 0.08220548182725906, 0.35256651043891907]",835,"[0.29914286732673645, 0.9049357771873474, 0.9012799859046936, 0.3619024157524109, 0.941291332244873, 0.07709825038909912, 0.5215959548950195, 0.852748453617096, 0.08220548182725906, 0.35256651043891907, 0.11538472026586533, 0.09900753945112228]",835


Genuine ADASYN-generated rows (up to 5 per oversampled Attack):


,ADASYN row,Attack,Label,Kept after Tomek,categories__Protocol_0,categories__Protocol_17,categories__Protocol_6
0,237403,Benign,0,True,0.0,0.0,1.0
1,237404,Benign,0,True,0.0,0.0,1.0
2,237405,Benign,0,True,0.0,0.0,1.0
3,237406,Benign,0,True,0.0,0.0,1.0
4,237407,Benign,0,True,0.0,0.0,1.0
5,245807,Theft,1,True,0.0,0.0,1.0
6,245808,Theft,1,True,0.0,0.0,1.0
7,245809,Theft,1,True,0.0,0.0,1.0
8,245810,Theft,1,True,0.0,0.0,1.0
9,245811,Theft,1,True,0.0,0.0,1.0


Generated non-binary indicator coordinates (first 40):


,ADASYN row,Attack,Label,Kept after Tomek,Encoded column,Generated value


No fractional indicator values occurred in this small preview. Use the full-data unique comparison above for all final rows.


### Baseline model and three partial-balancing policies

Enable RUN_THRESHOLD_TESTS to compare an unresampled baseline with selection=target ratios 0.25, 0.5 and 0.75. These are paired policy comparisons, not a selection-only ablation. All cases use the same FULL prepared training input and untouched prepared validation set. Attack is the multiclass target; neither Attack nor Label enters the features. Test/zero-day data is not read.

Each fresh model uses one 32-unit ReLU layer, Adam, learning rate 0.001, alpha 1e-4, seed SEED and 30 epochs. The epoch budget is equal; larger training sets have more optimizer updates per epoch. Reports include accuracy, balanced accuracy, macro/weighted F1, per-Attack precision/recall/F1, validation support and confusion matrices. Macro F1 uses the fixed training-class list; inspect support for classes absent from validation.

Counts, generated rows, post-ADASYN matrix GiB (not process peak RAM), removed rows, sampler/training times and compute total are saved. Checkpoint/result reuse is explicit; cached durations describe original compute. Each policy owns a compatible checkpoint, including reuse of the main 0.5 ADASYN output. Successful models/reports are reused only when source train/validation, policy and model settings match. Errors are saved without substituting another sampler. These tests are optional and can be expensive.

Outputs in threshold_tests include comparison.csv, per-case model.joblib, metrics.json, class counts, generated previews and CSV/HTML confusion matrices. HTML plots use Plotly CDN. Choose thresholds on validation outcomes; reserve test data for a final evaluation.

Elapsed times in tables, CSVs, metrics JSON and prints use hours/minutes/seconds. Numeric seconds stay inside checkpoints for arithmetic; compatible older result files with second-based metrics are converted when reused.


In [45]:
if RUN_THRESHOLD_TESTS:
    import gc
    import json
    import joblib
    import numpy as np
    import pandas as pd
    import plotly.express as px
    import pyarrow.parquet as pq
    from time import perf_counter
    from sklearn.neural_network import MLPClassifier
    from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                                 classification_report, confusion_matrix)
    from imblearn.over_sampling import ADASYN
    from imblearn.under_sampling import TomekLinks

    THRESHOLDS = [0.25, 0.50, 0.75]  # Selection AND target move together.
    THRESHOLD_NN_EPOCHS = 30
    REUSE_THRESHOLD_RESULTS = True
    test_dir = OUTPUT_DIR / 'threshold_tests'
    test_dir.mkdir(parents=True, exist_ok=True)
    X_base, y_base, test_features, train_metadata = read_prepared_matrix(OUTPUT_DIR / 'train.parquet')
    X_validation, y_validation, _, _ = read_prepared_matrix(
        OUTPUT_DIR / 'val.parquet', test_features,
    )
    test_classes = np.unique(y_base)
    test_before = pd.Series(y_base).value_counts().sort_index()
    test_majority = test_before.idxmax()
    test_target_dtypes = train_metadata['target_dtypes']
    test_label_map = train_metadata['attack_to_label']
    del train_metadata
    # Recheck training content if the prepared file changed since main resampling.
    test_policy_settings = {**current_policy_settings,
                            'train_sha256': prepared_signature(OUTPUT_DIR / 'train.parquet'),
                            'features': test_features}
    validation_hash = prepared_signature(OUTPUT_DIR / 'val.parquet')
    threshold_rows = []

    for ratio in [None] + THRESHOLDS:
        case = 'baseline' if ratio is None else f'ratio_{ratio:g}'
        case_dir = test_dir / case
        case_dir.mkdir(parents=True, exist_ok=True)
        settings = {**test_policy_settings, 'selection': ratio or SELECT_BELOW_RATIO,
                    'target': ratio or TARGET_RATIO}
        model_settings = {
            'policy': settings, 'baseline': ratio is None, 'validation_sha256': validation_hash,
            'hidden': [32], 'epochs': THRESHOLD_NN_EPOCHS, 'seed': SEED,
            'learning_rate': 0.001, 'alpha': 1e-4,
        }
        metrics_path = case_dir / 'metrics.json'
        case_started = perf_counter()
        if REUSE_THRESHOLD_RESULTS and metrics_path.exists():
            previous = json.loads(metrics_path.read_text())
            required = ['model.joblib', 'per_attack_report.csv', 'class_counts.csv',
                        'confusion_matrix.csv', 'confusion_matrix.html']
            if previous['settings'] == model_settings and all((case_dir / name).exists() for name in required):
                row = previous['metrics'].copy()
                # Older saved results use numeric second columns; display them as h/m/s too.
                for name in list(row):
                    if name.endswith(' seconds'):
                        row[name.removesuffix(' seconds') + ' time'] = format_duration(row.pop(name))
                row['Result reused'] = True
                row['Original run wall time'] = row.get('Original run wall time', row['This run wall time'])
                row['This run wall time'] = format_duration(perf_counter() - case_started)
                metrics_path.write_text(json.dumps({'settings': model_settings, 'metrics': row}, indent=2))
                threshold_rows.append(row)
                print('Reusing completed test:', case, '(stored compute times)', flush=True)
                continue
        wall_started = perf_counter()
        state = None
        try:
            print(f'Threshold test: {case}', flush=True)
            adasyn_time = tomek_time = 0.0
            adasyn_cached = False
            adasyn_backend_used = 'not used'
            requested_counts = test_before.to_dict()
            if ratio is None:
                X_fit, y_fit = X_base, y_base
                after_adasyn = test_before.copy()
                adasyn_matrix_gib = X_base.nbytes / 2**30
                added = removed = 0
            else:
                requested = math.ceil(ratio * int(test_before.max()))
                targets = {attack: requested for attack, count in test_before.items()
                           if count < ratio * test_before.max()
                           and count < (1 - ADASYN_BALANCE_TOLERANCE) * requested}
                requested_counts.update(targets)
                policy_dir = OUTPUT_DIR / 'resampling' / f'select_{ratio:g}_target_{ratio:g}'
                checkpoint = policy_dir / 'adasyn_checkpoint.joblib'
                if REUSE_ADASYN_CHECKPOINT and checkpoint.exists():
                    payload = joblib.load(checkpoint, mmap_mode='r')
                    if payload['settings'] != settings:
                        raise ValueError('Checkpoint settings/data differ. Set REUSE_ADASYN_CHECKPOINT=False to recompute.')
                    state = payload['state']
                    del payload
                    adasyn_cached = True
                else:
                    started = perf_counter()
                    if targets:
                        smallest = min(int(test_before[a]) for a in targets)
                        if smallest < 2:
                            raise ValueError('ADASYN needs at least two training examples per targeted Attack.')
                        print(f'{case} ADASYN: starting...', flush=True)
                        X_after, y_after = ADASYN(
                            sampling_strategy=targets, n_neighbors=min(5, smallest - 1),
                            random_state=SEED,
                        ).fit_resample(X_base, y_base)
                        adasyn_time = perf_counter() - started
                    else:
                        X_after, y_after = X_base, y_base
                    state = {
                        'X_adasyn': X_after, 'y_adasyn': y_after,
                        'resampling_features': test_features,
                        'original_columns': pq.read_schema(OUTPUT_DIR / 'train.parquet').names,
                        'target_dtypes': test_target_dtypes, 'attack_to_label': test_label_map,
                        'before_counts': test_before, 'adasyn_counts': pd.Series(y_after).value_counts().sort_index(),
                        'majority_attack': test_majority, 'input_train_rows': len(y_base),
                        'adasyn_rows': len(y_after), 'adasyn_seconds': adasyn_time,
                        'adasyn_skipped': not bool(targets), 'sampling_targets': targets,
                        'adasyn_backend': GPU_STATUS if targets else 'skipped',
                        'generated_preview': generated_row_preview(
                            X_after, y_after, len(y_base), targets, test_features, test_label_map,
                        ),
                    }
                    atomic_checkpoint(checkpoint, state, settings)
                    del X_after, y_after
                X_after, y_after = state['X_adasyn'], state['y_adasyn']
                adasyn_time = state['adasyn_seconds']
                adasyn_backend_used = state['adasyn_backend']
                adasyn_matrix_gib = X_after.nbytes / 2**30
                added = len(y_after) - len(y_base)
                after_adasyn = pd.Series(y_after).value_counts().sort_index()
                started = perf_counter()
                print(f'{case} Tomek Links: starting...', flush=True)
                case_tomek = TomekLinks(sampling_strategy=[test_majority], n_jobs=-1)
                X_fit, y_fit = case_tomek.fit_resample(X_after, y_after)
                tomek_time = perf_counter() - started
                removed = len(y_after) - len(y_fit)
                preview = state['generated_preview'].copy()
                preview['Kept after Tomek'] = np.isin(preview['ADASYN row'], case_tomek.sample_indices_)
                preview_path = case_dir / 'generated_preview.parquet'
                preview.to_parquet(preview_path, index=False)
                del state, X_after, y_after, preview
                state = None
            counts = pd.DataFrame({
                'Before': test_before, 'Requested': pd.Series(requested_counts),
                'After ADASYN': after_adasyn, 'After Tomek': pd.Series(y_fit).value_counts(),
            }).fillna(0).astype(int).rename_axis('Attack')
            counts_path = case_dir / 'class_counts.csv'
            counts.to_csv(counts_path)
            print(f'{case}: training identical 32-unit MLP for {THRESHOLD_NN_EPOCHS} epochs; '
                  f'X={X_fit.shape}', flush=True)
            model = MLPClassifier(
                hidden_layer_sizes=(32,), activation='relu', solver='adam',
                batch_size=min(1024, len(y_fit)), learning_rate_init=0.001,
                alpha=1e-4, random_state=SEED,
            )
            started = perf_counter()
            for epoch in range(THRESHOLD_NN_EPOCHS):
                model.partial_fit(X_fit, y_fit, classes=test_classes)
            train_time = perf_counter() - started
            prediction = model.predict(X_validation)
            row = {
                'Case': case, 'Status': 'finished', 'Selection': ratio, 'Target': ratio,
                'Training rows': len(y_fit), 'Generated rows': added, 'Tomek removed': removed,
                'Post-ADASYN GiB': adasyn_matrix_gib, 'ADASYN time': format_duration(adasyn_time),
                'Tomek time': format_duration(tomek_time), 'Training time': format_duration(train_time),
                'Compute time': format_duration(adasyn_time + tomek_time + train_time),
                'Accuracy': float(accuracy_score(y_validation, prediction)),
                'Balanced accuracy': float(balanced_accuracy_score(y_validation, prediction)),
                'Macro F1': float(f1_score(y_validation, prediction, labels=test_classes, average='macro', zero_division=0)),
                'Weighted F1': float(f1_score(y_validation, prediction, labels=test_classes, average='weighted', zero_division=0)),
                'ADASYN cached': adasyn_cached, 'ADASYN backend': adasyn_backend_used,
                'Tomek backend': GPU_STATUS if ratio is not None else 'not used',
                'Result reused': False, 'This run wall time': format_duration(perf_counter() - wall_started),
            }
            report = pd.DataFrame(classification_report(
                y_validation, prediction, labels=test_classes, output_dict=True, zero_division=0,
            )).T
            report_path = case_dir / 'per_attack_report.csv'
            report.to_csv(report_path)
            matrix = confusion_matrix(y_validation, prediction, labels=test_classes)
            matrix_path = case_dir / 'confusion_matrix.csv'
            pd.DataFrame(matrix, index=test_classes, columns=test_classes).to_csv(matrix_path)
            fig = px.imshow(matrix, x=test_classes, y=test_classes, text_auto=True,
                            labels={'x': 'Predicted Attack', 'y': 'True Attack'}, title=case)
            figure_path = case_dir / 'confusion_matrix.html'
            fig.write_html(figure_path, include_plotlyjs='cdn')
            fig.show()
            model_path = case_dir / 'model.joblib'
            joblib.dump(model, model_path)
            metrics_path.write_text(json.dumps({'settings': model_settings, 'metrics': row}, indent=2))
            threshold_rows.append(row)
        except (ValueError, RuntimeError, MemoryError) as error:
            # A failed sampler is reported, never silently replaced by another method.
            row = {'Case': case, 'Status': 'failed', 'Error': str(error), 'Result reused': False}
            threshold_rows.append(row)
            error_path = case_dir / 'error.json'
            error_path.write_text(json.dumps(row, indent=2))
            print(case, 'failed:', error, flush=True)
        finally:
            state = None
            for variable in ['X_fit', 'y_fit', 'X_after', 'y_after', 'model', 'case_tomek']:
                globals().pop(variable, None)
            gc.collect()
        # Keep successful partial results if a later configuration fails.
        threshold_results = pd.DataFrame(threshold_rows)
        summary_path = test_dir / 'comparison.csv'
        threshold_results.to_csv(summary_path, index=False)

    threshold_results = pd.DataFrame(threshold_rows)
    summary_path = test_dir / 'comparison.csv'
    threshold_results.to_csv(summary_path, index=False)
    display(threshold_results)
    class_count_frames = []
    for case in threshold_results['Case']:
        path = test_dir / case / 'class_counts.csv'
        if path.exists():
            frame = pd.read_csv(path)
            frame['Case'] = case
            class_count_frames.append(frame)
    if class_count_frames:
        count_comparison = pd.concat(class_count_frames, ignore_index=True)
        fig = px.bar(count_comparison, x='Attack', y='After Tomek', color='Case',
                     barmode='group', title='Training class counts: baseline and threshold policies')
        path = test_dir / 'class_distribution.html'
        fig.write_html(path, include_plotlyjs='cdn')
        fig.show()
    del X_base, y_base, X_validation, y_validation
    gc.collect()


Loading 237,403 rows × 79 features; dtype=float32; working arrays ≈ 0.07 GiB; available RAM=29.58 GiB
Loading 50,872 rows × 79 features; dtype=float32; working arrays ≈ 0.02 GiB; available RAM=29.58 GiB
Threshold test: baseline
baseline: training identical 32-unit MLP for 30 epochs; X=(237403, 79)


Threshold test: ratio_0.25
ratio_0.25 ADASYN: starting...
Checkpoint saved: /kaggle/working/project/outputs/CIC-BoT-IoT/preprocessing/20261116_112425_all_attacks_IterativeImputer/resampling/select_0.25_target_0.25/adasyn_checkpoint.joblib
ratio_0.25 Tomek Links: starting...
ratio_0.25: training identical 32-unit MLP for 30 epochs; X=(231474, 79)


Threshold test: ratio_0.5
ratio_0.5 Tomek Links: starting...
ratio_0.5: training identical 32-unit MLP for 30 epochs; X=(257409, 79)


Threshold test: ratio_0.75
ratio_0.75 ADASYN: starting...
Checkpoint saved: /kaggle/working/project/outputs/CIC-BoT-IoT/preprocessing/20261116_112425_all_attacks_IterativeImputer/resampling/select_0.75_target_0.75/adasyn_checkpoint.joblib
ratio_0.75 Tomek Links: starting...
ratio_0.75: training identical 32-unit MLP for 30 epochs; X=(292746, 79)


,Case,Status,Selection,Target,Training rows,Generated rows,Tomek removed,Post-ADASYN GiB,ADASYN time,Tomek time,...,Compute time,Accuracy,Balanced accuracy,Macro F1,Weighted F1,ADASYN cached,ADASYN backend,Tomek backend,Result reused,This run wall time
0,baseline,finished,NaN,NaN,237403,0,0,0.069867,00h 00m 00.0s,00h 00m 00.0s,...,00h 00m 38.1s,0.620164,0.648768,0.609173,0.529116,False,not used,not used,False,00h 00m 38.6s
1,ratio_0.25,finished,0.25,0.25,231474,16930,22859,0.074850,00h 00m 01.6s,00h 05m 09.4s,...,00h 05m 46.5s,0.618140,0.721734,0.576678,0.520140,False,CPU (NVIDIA GPU/CuPy unavailable),CPU (NVIDIA GPU/CuPy unavailable),False,00h 05m 47.1s
2,ratio_0.5,finished,0.50,0.50,257409,42868,22862,0.082483,00h 00m 35.0s,00h 06m 12.9s,...,00h 07m 28.1s,0.617589,0.724450,0.573209,0.518894,True,CPU (NVIDIA GPU/CuPy unavailable),CPU (NVIDIA GPU/CuPy unavailable),False,00h 06m 53.8s
3,ratio_0.75,finished,0.75,0.75,292746,78209,22866,0.092884,00h 00m 35.4s,00h 08m 01.4s,...,00h 09m 24.3s,0.615466,0.721292,0.583551,0.518282,False,CPU (NVIDIA GPU/CuPy unavailable),CPU (NVIDIA GPU/CuPy unavailable),False,00h 09m 24.9s
